# YOLOv8-OBB Training — Pick & Place boxes

**Перед запуском:**
1. Runtime → Change runtime type → **T4 GPU**
2. Загрузи `dataset.zip` и `train_colab.ipynb` в папку `Diploma/` на Google Drive
3. Запускай ячейки по порядку

## 1. Установка зависимостей

In [ ]:
!pip install ultralytics --quiet
import ultralytics
ultralytics.checks()  # покажет версию и доступность GPU

## 2. Подключение Google Drive и распаковка датасета

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile

ZIP_PATH    = '/content/drive/MyDrive/Diploma/dataset.zip'
EXTRACT_DIR = '/content/dataset'
DATASET_DIR = '/content/dataset'
DATASET_YAML = f'{DATASET_DIR}/dataset.yaml'

# Распаковываем zip в /content/ (быстрее чем читать с Drive)
if not os.path.exists(DATASET_YAML):
    print('Распаковываю dataset.zip ...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/')
    print('Готово.')
else:
    print('Датасет уже распакован.')

assert os.path.exists(DATASET_YAML), f'dataset.yaml не найден: {DATASET_YAML}'
print('dataset.yaml найден ✓')

from pathlib import Path
n_train = len(list(Path(DATASET_DIR, 'images', 'train').glob('*.png')))
n_val   = len(list(Path(DATASET_DIR, 'images', 'val').glob('*.png')))
print(f'train: {n_train}  val: {n_val}  total: {n_train + n_val}')

## 3. Обновление путей в dataset.yaml

In [ ]:
# Перезаписываем yaml с актуальным путём внутри Colab
yaml_content = f"""path: {DATASET_DIR}
train: images/train
val:   images/val

nc: 1
names: ['box']
"""

with open(DATASET_YAML, 'w') as f:
    f.write(yaml_content)

print('dataset.yaml обновлён:')
print(yaml_content)

## 4. Обучение YOLOv8n-OBB

In [ ]:
!yolo train \
    model=yolov8n-obb.pt \
    data={DATASET_YAML} \
    epochs=50 \
    imgsz=640 \
    batch=16 \
    device=0 \
    project='/content/runs' \
    name='boxes_obb' \
    patience=15 \
    hsv_h=0.015 \
    hsv_s=0.3 \
    hsv_v=0.2 \
    degrees=3.0 \
    flipud=0.5 \
    fliplr=0.5 \
    mosaic=0.5 \
    plots=True

## 5. Результаты обучения

In [ ]:
from IPython.display import Image as IPImage, display

results_dir = '/content/runs/boxes_obb'

for plot in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    path = f'{results_dir}/{plot}'
    if os.path.exists(path):
        print(f'\n--- {plot} ---')
        display(IPImage(path, width=800))

## 6. Проверка метрик

In [ ]:
import pandas as pd

csv_path = f'{results_dir}/results.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    best_epoch = df['metrics/mAP50(B)'].idxmax()
    best = df.loc[best_epoch]

    print(f'Лучшая эпоха: {int(best_epoch) + 1}')
    print(f'  mAP50:      {best["metrics/mAP50(B)"]:.4f}')
    print(f'  mAP50-95:   {best["metrics/mAP50-95(B)"]:.4f}')
    print(f'  Precision:  {best["metrics/precision(B)"]:.4f}')
    print(f'  Recall:     {best["metrics/recall(B)"]:.4f}')

## 7. Сохранение best.pt на Google Drive

In [ ]:
import shutil

best_src = f'{results_dir}/weights/best.pt'
best_dst = '/content/drive/MyDrive/Diploma/yolo_boxes.pt'

shutil.copy(best_src, best_dst)
print(f'Сохранено: {best_dst} ✓')
print()
print('Следующий шаг — скачать yolo_boxes.pt с Google Drive на macOS')
print('и положить в: /Users/polina/Documents/Stankin_3/Diplom/worlds/models/')

## 8. Быстрая проверка на тестовом изображении

In [ ]:
from ultralytics import YOLO
import glob, random

model = YOLO(best_src)

val_imgs = glob.glob(f'{DATASET_DIR}/images/val/*.png')
test_img = random.choice(val_imgs)

results = model(test_img, conf=0.3)
results[0].save(fname='/content/test_result.jpg')

display(IPImage('/content/test_result.jpg', width=640))
print(f'Детектировано объектов: {len(results[0].obb)}')